In [20]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [21]:
# Подготовка данных (пример)
X = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_for_lstm_road_version_4.csv')
S = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_for_lstm_speed_version_4.csv')
y = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\y_train_for_lstm_wheel_4.csv')

In [22]:
X_train = torch.tensor(X.values, dtype=torch.float32).to(device='cuda')
S_train = torch.tensor(S.values, dtype=torch.float32).to(device='cuda')
y_tensor = torch.tensor(y.values, dtype=torch.float32).to(device='cuda')

In [23]:
print(len(X_train))
print(len(S_train))
print(len(y_tensor))

13037
13037
13037


In [24]:
X_tensor = torch.cat((X_train, S_train), dim=1)
print(X_tensor[0])
print(X_tensor[0].size())

tensor([ 0.,  0.,  0.,  ...,  0.,  0., 59.], device='cuda:0')
torch.Size([12289])


In [25]:
print(y_tensor[0])
print(y_tensor[0].size())

tensor([0.0053, 0.0053], device='cuda:0')
torch.Size([2])


In [29]:
dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=1, shuffle=False)

In [30]:
class LSTMNet(nn.Module):
    def __init__(self, input_size=12289, hidden_size=1024, output_size=2, num_layers=1):
        super(LSTMNet, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        
        self.linear = nn.Linear(hidden_size, output_size)
        
        self.tanh = nn.Tanh()
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).cuda()
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).cuda()
        
        out, _ = self.lstm(x, (h0, c0))
        
        out = out[:, -1, :]
        
        out = self.linear(out)
        
        out = self.tanh(out)
        
        return out


In [31]:
model_lstm = LSTMNet().to('cuda')
criterion = nn.MSELoss()  # Используйте подходящую функцию потерь
optimizer = optim.Adam(model_lstm.parameters(), lr=0.000001)

num_epochs = 10
for epoch in range(num_epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        outputs = model_lstm(X_batch)
        
        loss = criterion(outputs, y_batch)
        
        loss.backward()
        
        optimizer.step()
    
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

RuntimeError: For unbatched 2-D input, hx and cx should also be 2-D but got (3-D, 3-D) tensors

In [ ]:
torch.save(model.state_dict(), 'C:\PycharmProjects\ETS_Autopilot\static\weight_model\weight_wheel_nn_lstm_3.pth')

In [ ]:
class LSTMNet(nn.Module):
    def __init__(self, input_size=12289, hidden_size=1024, output_size=2):
        super(LSTMNet, self).__init__()
        
        self.hidden_size = hidden_size
        
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        
        self.linear = nn.Linear(hidden_size, output_size)
        
        self.tanh = nn.Tanh()
        
    def forward(self, x):
        h0 = torch.zeros(1, x.size(0), self.hidden_size).cuda()
        c0 = torch.zeros(1, x.size(0), self.hidden_size).cuda()
        
        out, _ = self.lstm(x, (h0, c0))
        
        out = out[:, -1, :]
        
        out = self.linear(out)
        
        out = self.tanh(out)
        
        return out